# نصب Chrome
sudo apt-get update
sudo apt-get install -y wget gnupg
wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | sudo apt-key add -
sudo sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
sudo apt-get update
sudo apt-get install -y google-chrome-stable

# نصب ChromeDriver
sudo apt-get install -y chromedriver

# بررسی نصب
chromedriver --version
google-chrome --version

In [ ]:
!pip install --upgrade pip

In [ ]:
!pip install selenium pandas webdriver-manager

In [24]:
import pandas as pd
import requests
from io import StringIO
import time

def get_bitcoin_transaction_fees_coinmetrics():
    """
    دریافت داده‌های روزانه هزینه تراکنش بیت‌کوین از Coin Metrics
    این داده‌ها شامل:
    - FeeMeanUSD: میانگین کارمزد هر تراکنش به دلار
    - FeeMedUSD: میانه کارمزد هر تراکنش به دلار
    - FeeMeanNtv: میانگین کارمزد هر تراکنش به ساتوشی
    """
    
    print("🔄 در حال دریافت داده‌های کارمزد تراکنش از Coin Metrics...")
    
    # آدرس فایل CSV عمومی Coin Metrics
    url = "https://raw.githubusercontent.com/coinmetrics/data/master/csv/btc.csv"
    
    try:
        # دانلود فایل CSV با timeout بیشتر
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        
        # خواندن داده‌ها
        df = pd.read_csv(StringIO(response.text))
        
        print(f"✅ فایل با موفقیت دانلود شد. تعداد کل رکوردها: {len(df)}")
        
        # پیدا کردن ستون‌های مرتبط با کارمزد
        fee_columns = [col for col in df.columns if any(keyword in col.lower() for keyword in ['fee', 'cost'])]
        
        if not fee_columns:
            print("⚠️ ستون کارمزد پیدا نشد. ستون‌های موجود:")
            print(df.columns.tolist())
            return None
        
        # انتخاب ستون‌های مورد نیاز
        columns_to_keep = ['time'] + fee_columns
        df_fees = df[columns_to_keep].copy()
        
        # تبدیل تاریخ به فرمت datetime
        df_fees['time'] = pd.to_datetime(df_fees['time'])
        
        # مرتب‌سازی بر اساس تاریخ
        df_fees = df_fees.sort_values('time').reset_index(drop=True)
        
        print(f"\n📊 ستون‌های کارمزد پیدا شد: {fee_columns}")
        print(f"📅 بازه زمانی: از {df_fees['time'].min()} تا {df_fees['time'].max()}")
        print(f"📈 تعداد رکوردها: {len(df_fees)}")
        
        return df_fees
        
    except requests.exceptions.Timeout:
        print("❌ خطا: زمان دانلود به پایان رسید. لطفاً اتصال اینترنت خود را بررسی کنید.")
        return None
    except requests.exceptions.RequestException as e:
        print(f"❌ خطا در دریافت فایل: {e}")
        return None
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        return None

def save_and_display_data(df):
    """ذخیره و نمایش داده‌ها"""
    if df is None or df.empty:
        print("❌ داده‌ای برای نمایش وجود ندارد.")
        return
    
    # نمایش ۱۰ رکورد اول
    print("\n" + "="*60)
    print("📊 ۱۰ رکورد اول داده‌های کارمزد تراکنش بیت‌کوین:")
    print("="*60)
    print(df.head(10).to_string())
    
    # نمایش آمار کلی
    print("\n" + "="*60)
    print("📈 آمار کلی:")
    print("="*60)
    print(f"میانگین کارمزد (USD): ${df['FeeMeanUSD'].mean():.2f}" if 'FeeMeanUSD' in df.columns else "ستون FeeMeanUSD وجود ندارد")
    print(f"حداکثر کارمزد (USD): ${df['FeeMeanUSD'].max():.2f}" if 'FeeMeanUSD' in df.columns else "")
    print(f"حداقل کارمزد (USD): ${df['FeeMeanUSD'].min():.2f}" if 'FeeMeanUSD' in df.columns else "")
    
    # ذخیره در فایل CSV
    try:
        filename = 'bitcoin_transaction_fees.csv'
        df.to_csv(filename, index=False)
        print(f"\n💾 داده‌ها با موفقیت در فایل '{filename}' ذخیره شدند.")
    except Exception as e:
        print(f"⚠️ خطا در ذخیره فایل: {e}")

# اجرای اصلی
if __name__ == "__main__":
    # دریافت داده‌ها
    fees_data = get_bitcoin_transaction_fees_coinmetrics()
    
    if fees_data is not None:
        # نمایش و ذخیره داده‌ها
        save_and_display_data(fees_data)
    else:
        print("\n❌ دریافت داده‌ها ناموفق بود. لطفاً روش‌های جایگزین را امتحان کنید.")

🔄 در حال دریافت داده‌های کارمزد تراکنش از Coin Metrics...
✅ فایل با موفقیت دانلود شد. تعداد کل رکوردها: 6351

📊 ستون‌های کارمزد پیدا شد: ['FeeTotNtv']
📅 بازه زمانی: از 2009-01-03 00:00:00 تا 2026-05-24 00:00:00
📈 تعداد رکوردها: 6351

📊 ۱۰ رکورد اول داده‌های کارمزد تراکنش بیت‌کوین:
        time  FeeTotNtv
0 2009-01-03        0.0
1 2009-01-04        0.0
2 2009-01-05        0.0
3 2009-01-06        0.0
4 2009-01-07        0.0
5 2009-01-08        0.0
6 2009-01-09        0.0
7 2009-01-10        0.0
8 2009-01-11        0.0
9 2009-01-12        0.0

📈 آمار کلی:
ستون FeeMeanUSD وجود ندارد



💾 داده‌ها با موفقیت در فایل 'bitcoin_transaction_fees.csv' ذخیره شدند.


In [1]:
import pandas as pd
import requests
from io import StringIO
from datetime import datetime

def get_bitcoin_fees_coinmetrics(start_date='2021-01-01', end_date='2025-12-02'):
    """
    دریافت داده‌های روزانه میانگین کارمزد هر تراکنش بیت‌کوین از Coin Metrics
    در بازه زمانی مشخص
    """
    
    print(f"🔄 در حال دریافت داده‌های کارمزد از Coin Metrics...")
    print(f"📅 بازه زمانی: از {start_date} تا {end_date}")
    
    # آدرس فایل CSV عمومی Coin Metrics
    url = "https://raw.githubusercontent.com/coinmetrics/data/master/csv/btc.csv"
    
    try:
        # دانلود فایل CSV
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        
        # خواندن داده‌ها
        df = pd.read_csv(StringIO(response.text))
        print(f"✅ فایل با موفقیت دانلود شد. تعداد کل رکوردها: {len(df)}")
        
        # پیدا کردن ستون‌های مربوط به میانگین کارمزد
        mean_fee_cols = [col for col in df.columns if 'mean' in col.lower() and 'fee' in col.lower()]
        
        # انتخاب ستون‌های مورد نیاز
        columns_to_keep = ['time']
        
        # اولویت: FeeMeanUSD (میانگین کارمزد به دلار)
        if 'FeeMeanUSD' in mean_fee_cols:
            columns_to_keep.append('FeeMeanUSD')
            print("✅ ستون FeeMeanUSD (میانگین کارمزد به دلار) پیدا شد.")
        elif 'FeeMeanNtv' in mean_fee_cols:
            columns_to_keep.append('FeeMeanNtv')
            print("✅ ستون FeeMeanNtv (میانگین کارمزد به ساتوشی) پیدا شد.")
        else:
            print("⚠️ ستون میانگین کارمزد پیدا نشد. ستون‌های موجود:")
            print([col for col in df.columns if 'fee' in col.lower()])
            return None
        
        # فیلتر کردن داده‌ها
        df_fees = df[columns_to_keep].copy()
        df_fees['time'] = pd.to_datetime(df_fees['time'])
        
        # فیلتر بر اساس بازه زمانی
        mask = (df_fees['time'] >= start_date) & (df_fees['time'] <= end_date)
        df_fees = df_fees.loc[mask].copy()
        
        # مرتب‌سازی بر اساس تاریخ
        df_fees = df_fees.sort_values('time').reset_index(drop=True)
        
        print(f"\n📊 تعداد رکوردها در بازه زمانی انتخاب شده: {len(df_fees)}")
        print(f"📅 از {df_fees['time'].min()} تا {df_fees['time'].max()}")
        
        # نمایش نمونه
        print("\n📊 ۱۰ رکورد اول:")
        print(df_fees.head(10).to_string())
        
        return df_fees
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return None

def save_and_analyze_data(df, filename='bitcoin_fees_2021_2025.csv'):
    """ذخیره و تحلیل داده‌ها"""
    
    if df is None or df.empty:
        print("❌ داده‌ای برای ذخیره وجود ندارد.")
        return
    
    # ذخیره در فایل CSV
    df.to_csv(filename, index=False)
    print(f"\n💾 داده‌ها در فایل '{filename}' ذخیره شدند.")
    
    # نمایش آمار
    print("\n" + "="*60)
    print("📈 آمار کلی کارمزد تراکنش بیت‌کوین (بازه انتخابی):")
    print("="*60)
    
    if 'FeeMeanUSD' in df.columns:
        print(f"میانگین کارمزد: ${df['FeeMeanUSD'].mean():.2f}")
        print(f"حداکثر کارمزد: ${df['FeeMeanUSD'].max():.2f}")
        print(f"حداقل کارمزد: ${df['FeeMeanUSD'].min():.2f}")
        print(f"انحراف معیار: ${df['FeeMeanUSD'].std():.2f}")
        
        # پیدا کردن روزهای خاص
        max_date = df.loc[df['FeeMeanUSD'].idxmax(), 'time']
        min_date = df.loc[df['FeeMeanUSD'].idxmin(), 'time']
        print(f"\n📌 بالاترین کارمزد: {df['FeeMeanUSD'].max():.2f} در تاریخ {max_date}")
        print(f"📌 پایین‌ترین کارمزد: {df['FeeMeanUSD'].min():.2f} در تاریخ {min_date}")
    
    # نمایش خلاصه آماری
    print("\n📊 خلاصه آماری:")
    print(df.describe())

# ========== اجرای اصلی ==========
if __name__ == "__main__":
    # دریافت داده‌ها
    df_fees = get_bitcoin_fees_coinmetrics(
        start_date='2021-01-01',
        end_date='2025-12-02'
    )
    
    if df_fees is not None:
        # ذخیره و تحلیل
        save_and_analyze_data(df_fees)
    else:
        print("\n❌ دریافت داده‌ها ناموفق بود.")

🔄 در حال دریافت داده‌های کارمزد از Coin Metrics...
📅 بازه زمانی: از 2021-01-01 تا 2025-12-02
✅ فایل با موفقیت دانلود شد. تعداد کل رکوردها: 6351
⚠️ ستون میانگین کارمزد پیدا نشد. ستون‌های موجود:
['FeeTotNtv']

❌ دریافت داده‌ها ناموفق بود.


In [2]:
def get_bitcoin_fees_fallback(start_date='2021-01-01', end_date='2025-12-02'):
    """روش جایگزین با انتخاب خودکار بهترین ستون"""
    
    url = "https://raw.githubusercontent.com/coinmetrics/data/master/csv/btc.csv"
    
    try:
        df = pd.read_csv(url)
        df['time'] = pd.to_datetime(df['time'])
        
        # فیلتر بر اساس بازه زمانی
        mask = (df['time'] >= start_date) & (df['time'] <= end_date)
        df_filtered = df.loc[mask].copy()
        
        # پیدا کردن ستون‌های کارمزد
        fee_cols = [col for col in df_filtered.columns if 'fee' in col.lower()]
        
        # اولویت با ستون‌های USD
        usd_fee_cols = [col for col in fee_cols if 'usd' in col.lower()]
        mean_fee_cols = [col for col in usd_fee_cols if 'mean' in col.lower()]
        med_fee_cols = [col for col in usd_fee_cols if 'med' in col.lower()]
        tot_fee_cols = [col for col in usd_fee_cols if 'tot' in col.lower()]
        
        # انتخاب بهترین ستون
        selected_cols = ['time']
        if mean_fee_cols:
            selected_cols.append(mean_fee_cols[0])
            print(f"✅ ستون انتخاب شده: {mean_fee_cols[0]}")
        elif med_fee_cols:
            selected_cols.append(med_fee_cols[0])
            print(f"✅ ستون انتخاب شده: {med_fee_cols[0]}")
        elif tot_fee_cols:
            selected_cols.append(tot_fee_cols[0])
            print(f"✅ ستون انتخاب شده: {tot_fee_cols[0]}")
        else:
            print("❌ هیچ ستون مناسبی پیدا نشد.")
            return None
        
        df_result = df_filtered[selected_cols].copy()
        df_result = df_result.sort_values('time').reset_index(drop=True)
        
        print(f"\n📊 تعداد رکوردها: {len(df_result)}")
        print(df_result.head(10))
        
        return df_result
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return None

# اجرا
df_fallback = get_bitcoin_fees_fallback()
if df_fallback is not None:
    df_fallback.to_csv('bitcoin_fees_alternative.csv', index=False)

❌ هیچ ستون مناسبی پیدا نشد.


In [3]:
import pandas as pd
import requests
from datetime import datetime
import time

def get_bitcoin_transaction_fees(start_date='2021-01-01', end_date='2025-12-02'):
    """
    دریافت داده‌های روزانه کل کارمزد تراکنش‌های بیت‌کوین (به BTC) از Blockchain.info
    و تبدیل به دلار با استفاده از قیمت لحظه‌ای
    """
    
    print(f"🔄 در حال دریافت داده‌های کارمزد از Blockchain.info...")
    print(f"📅 بازه زمانی: از {start_date} تا {end_date}")
    
    # تبدیل تاریخ به تایم‌استمپ
    start_ts = int(pd.Timestamp(start_date).timestamp())
    end_ts = int(pd.Timestamp(end_date).timestamp())
    
    # دریافت داده‌های کارمزد
    url_fees = "https://api.blockchain.info/charts/transaction-fees"
    params = {
        'format': 'json',
        'timespan': 'all',
        'start': start_ts,
        'end': end_ts
    }
    
    try:
        # دریافت داده‌های کارمزد
        print("📡 در حال دریافت داده‌های کارمزد...")
        response_fees = requests.get(url_fees, params=params, timeout=30)
        response_fees.raise_for_status()
        data_fees = response_fees.json()
        
        # دریافت داده‌های قیمت بیت‌کوین
        print("📡 در حال دریافت داده‌های قیمت بیت‌کوین...")
        url_price = "https://api.blockchain.info/charts/market-price"
        params_price = {
            'format': 'json',
            'timespan': 'all',
            'start': start_ts,
            'end': end_ts
        }
        response_price = requests.get(url_price, params=params_price, timeout=30)
        response_price.raise_for_status()
        data_price = response_price.json()
        
        # ساخت دیتافریم کارمزد
        df_fees = pd.DataFrame(data_fees['values'])
        df_fees['date'] = pd.to_datetime(df_fees['x'], unit='s')
        df_fees['fee_btc'] = df_fees['y']  # کارمزد به بیت‌کوین
        
        # ساخت دیتافریم قیمت
        df_price = pd.DataFrame(data_price['values'])
        df_price['date'] = pd.to_datetime(df_price['x'], unit='s')
        df_price['price_usd'] = df_price['y']  # قیمت به دلار
        
        # ادغام دو دیتافریم
        df = pd.merge(df_fees[['date', 'fee_btc']], df_price[['date', 'price_usd']], on='date', how='inner')
        
        # محاسبه کارمزد به دلار
        df['fee_usd'] = df['fee_btc'] * df['price_usd']
        
        # مرتب‌سازی بر اساس تاریخ
        df = df.sort_values('date').reset_index(drop=True)
        
        # محاسبه میانگین کارمزد هر تراکنش (تخمینی)
        # برای این کار به تعداد تراکنش‌ها نیاز داریم
        print("📡 در حال دریافت داده‌های تعداد تراکنش‌ها...")
        url_tx = "https://api.blockchain.info/charts/n-transactions"
        params_tx = {
            'format': 'json',
            'timespan': 'all',
            'start': start_ts,
            'end': end_ts
        }
        response_tx = requests.get(url_tx, params=params_tx, timeout=30)
        response_tx.raise_for_status()
        data_tx = response_tx.json()
        
        df_tx = pd.DataFrame(data_tx['values'])
        df_tx['date'] = pd.to_datetime(df_tx['x'], unit='s')
        df_tx['tx_count'] = df_tx['y']
        
        # ادغام با داده‌های قبلی
        df = pd.merge(df, df_tx[['date', 'tx_count']], on='date', how='inner')
        
        # محاسبه میانگین کارمزد هر تراکنش
        df['avg_fee_usd'] = df['fee_usd'] / df['tx_count']
        df['avg_fee_btc'] = df['fee_btc'] / df['tx_count']
        
        # حذف مقادیر بی‌نهایت و NaN
        df = df.replace([float('inf'), float('-inf')], float('nan'))
        df = df.dropna()
        
        print(f"\n✅ دریافت {len(df)} رکورد از Blockchain.info")
        print(f"📅 از {df['date'].min()} تا {df['date'].max()}")
        
        # نمایش نمونه
        print("\n📊 ۱۰ رکورد اول:")
        print(df.head(10).to_string())
        
        return df
        
    except Exception as e:
        print(f"❌ خطا: {e}")
        return None

def save_and_analyze(df, filename='bitcoin_transaction_fees_2021_2025.csv'):
    """ذخیره و تحلیل داده‌ها"""
    
    if df is None or df.empty:
        print("❌ داده‌ای برای ذخیره وجود ندارد.")
        return
    
    # ذخیره در فایل CSV
    df.to_csv(filename, index=False)
    print(f"\n💾 داده‌ها در فایل '{filename}' ذخیره شدند.")
    
    # نمایش آمار
    print("\n" + "="*60)
    print("📈 آمار کلی کارمزد تراکنش بیت‌کوین (بازه انتخابی):")
    print("="*60)
    
    if 'avg_fee_usd' in df.columns:
        print(f"میانگین کارمزد هر تراکنش: ${df['avg_fee_usd'].mean():.2f}")
        print(f"حداکثر کارمزد هر تراکنش: ${df['avg_fee_usd'].max():.2f}")
        print(f"حداقل کارمزد هر تراکنش: ${df['avg_fee_usd'].min():.2f}")
        print(f"انحراف معیار: ${df['avg_fee_usd'].std():.2f}")
        
        # پیدا کردن روزهای خاص
        max_date = df.loc[df['avg_fee_usd'].idxmax(), 'date']
        min_date = df.loc[df['avg_fee_usd'].idxmin(), 'date']
        print(f"\n📌 بالاترین کارمزد: ${df['avg_fee_usd'].max():.2f} در تاریخ {max_date}")
        print(f"📌 پایین‌ترین کارمزد: ${df['avg_fee_usd'].min():.2f} در تاریخ {min_date}")
    
    # نمایش خلاصه آماری
    print("\n📊 خلاصه آماری:")
    print(df[['avg_fee_usd', 'price_usd', 'tx_count']].describe())

# ========== اجرای اصلی ==========
if __name__ == "__main__":
    # دریافت داده‌ها
    df = get_bitcoin_transaction_fees(
        start_date='2021-01-01',
        end_date='2025-12-02'
    )
    
    if df is not None:
        # ذخیره و تحلیل
        save_and_analyze(df)
    else:
        print("\n❌ دریافت داده‌ها ناموفق بود.")
        print("\n💡 راه‌حل جایگزین: استفاده از داده‌های آماده از Kaggle یا CoinGecko")

🔄 در حال دریافت داده‌های کارمزد از Blockchain.info...
📅 بازه زمانی: از 2021-01-01 تا 2025-12-02
📡 در حال دریافت داده‌های کارمزد...
📡 در حال دریافت داده‌های قیمت بیت‌کوین...
📡 در حال دریافت داده‌های تعداد تراکنش‌ها...

✅ دریافت 1999 رکورد از Blockchain.info
📅 از 2021-01-01 00:00:00 تا 2026-06-25 00:00:00

📊 ۱۰ رکورد اول:
        date     fee_btc  price_usd       fee_usd  tx_count  avg_fee_usd  avg_fee_btc
0 2021-01-01   48.942701   28982.56  1.418485e+06  258080.0     5.496299     0.000190
1 2021-01-02   79.699067   29393.75  2.342654e+06  297111.0     7.884779     0.000268
2 2021-01-03   86.419994   32195.46  2.782331e+06  359116.0     7.747723     0.000241
3 2021-01-04  108.705036   33000.78  3.587351e+06  373734.0     9.598674     0.000291
4 2021-01-05  113.365867   32035.03  3.631679e+06  354091.0    10.256344     0.000320
5 2021-01-06  128.518985   34046.67  4.375643e+06  397384.0    11.011121     0.000323
6 2021-01-07  121.912745   36860.41  4.493754e+06  401744.0    11.185615    